In [12]:
import requests
import pandas as pd

BASE = "https://www.airport.co.kr"

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0",
    "Referer": BASE + "/www/cms/frCon/index.do?MENU_ID=1250",  # 항공통계 페이지(대략)
})

# 1) 먼저 페이지 접속(쿠키 세팅용)
session.get(BASE + "/www/cms/frCon/index.do?MENU_ID=1250", timeout=30)

# 2) 실제 데이터 요청 (statisForm serialize 값)
url = BASE + "/www/ajaxf/frFlightStatsSvc/airLineStatsList.do"

payload = {
    # ✅ 아래 키들은 statisForm 안의 name 값과 1:1로 맞아야 합니다.
    # HTML에 보이는 것들 기준으로 “ST_YY, ST_MM, EN_YY, EN_MM”는 확실히 존재합니다.
    "ST_YY": "2024",
    "ST_MM": "01",
    "EN_YY": "2024",
    "EN_MM": "01",

    # 아래는 페이지에서 라디오로 선택되는 값들(코드에 name이 보입니다)
    # 실제 value(C4101 등)는 화면에서 선택한 값과 동일하게 넣어야 합니다.
    "PASS_TYPE": "C4101",   # 예: 유임여객
    "CAGO_TYPE": "C4201",   # 예: 화물
    # "LINE_TYPE": "...",
    # "RAGUL_TYPE": "...",
    # "USE_TYPE": "...",
    # "AL_TYPE": "...",
    # "AIRPORT_CHK": ["...","..."]  # 체크박스면 리스트로 들어갈 수 있음(serialize 방식 확인 필요)
}

r = session.post(url, data=payload, timeout=30)
r.raise_for_status()

data = r.json()  # 리스트(JSON 배열) 형태일 가능성이 큼
df = pd.DataFrame(data)

print(df.head())
print(df.columns)


                                                data
0  {'A_AIRLINE': 'AAR', 'A_AIRKOR': '아시아나항공', 'S_...
1  {'A_AIRLINE': 'ABL', 'A_AIRKOR': '에어부산', 'S_OC...
2  {'A_AIRLINE': 'ANA', 'A_AIRKOR': '전일본공수', 'S_O...
3  {'A_AIRLINE': 'ASV', 'A_AIRKOR': '에어서울', 'S_OC...
4  {'A_AIRLINE': 'BAV', 'A_AIRKOR': '벰부항공', 'S_OC...
Index(['data'], dtype='object')


In [31]:
import requests
import pandas as pd
from datetime import datetime
from dateutil.relativedelta import relativedelta
import time

BASE = "https://www.airport.co.kr"

LIST_URL = BASE + "/www/ajaxf/frFlightStatsSvc/airLineStatsList.do"
ENTRY_URL = BASE + "/www/cms/frCon/index.do?MENU_ID=1250"


def month_range(start_yyyymm: str, end_yyyymm: str):
    """YYYYMM → generator"""
    cur = datetime.strptime(start_yyyymm, "%Y%m")
    end = datetime.strptime(end_yyyymm, "%Y%m")
    while cur <= end:
        yield cur.strftime("%Y%m")
        cur += relativedelta(months=1)


def fetch_one_month(session: requests.Session, yyyymm: str) -> pd.DataFrame:
    yyyy = yyyymm[:4]
    mm = yyyymm[4:]

    payload = {
        "ST_YY": yyyy,
        "ST_MM": mm,
        "EN_YY": yyyy,
        "EN_MM": mm,

        # ⚠️ 이 값들은 Network 탭에서 확인된 기본값 기준
        # 필요 시 변경 가능
        "PASS_TYPE": "C4101",   # 유임여객
        "CAGO_TYPE": "C4201",   # 화물
    }

    r = session.post(LIST_URL, data=payload, timeout=30)
    r.raise_for_status()

    data = r.json()          # list[dict]
    if not data:
        return pd.DataFrame()

    df = pd.json_normalize(data)
    df["yyyymm"] = yyyymm
    df["date"] = pd.to_datetime(yyyymm, format="%Y%m") + pd.offsets.MonthEnd(0)

    return df


def fetch_airline_stats_72months(
    start_yyyymm="202001",
    end_yyyymm="202512",
    sleep_sec=0.3,
) -> pd.DataFrame:

    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Referer": ENTRY_URL,
    })

    # 1️⃣ 초기 진입 (쿠키 세팅)
    session.get(ENTRY_URL, timeout=30)

    frames = []

    for yyyymm in month_range(start_yyyymm, end_yyyymm):
        try:
            print(f"[FETCH] {yyyymm}")
            df_m = fetch_one_month(session, yyyymm)
            if not df_m.empty:
                frames.append(df_m)
            time.sleep(sleep_sec)
        except Exception as e:
            print(f"[WARN] {yyyymm} failed: {e}")

    if not frames:
        return pd.DataFrame()

    df_all = pd.concat(frames, ignore_index=True)

    return df_all



In [32]:
df_air = fetch_airline_stats_72months(
    start_yyyymm="202001",
    end_yyyymm="202512"
)

print(df_air.shape)
print(df_air.head())
print(df_air.tail())

[FETCH] 202001
[FETCH] 202002
[FETCH] 202003
[FETCH] 202004
[FETCH] 202005
[FETCH] 202006
[FETCH] 202007
[FETCH] 202008
[FETCH] 202009
[FETCH] 202010
[FETCH] 202011
[FETCH] 202012
[FETCH] 202101
[FETCH] 202102
[FETCH] 202103
[FETCH] 202104
[FETCH] 202105
[FETCH] 202106
[FETCH] 202107
[FETCH] 202108
[FETCH] 202109
[FETCH] 202110
[FETCH] 202111
[FETCH] 202112
[FETCH] 202201
[FETCH] 202202
[FETCH] 202203
[FETCH] 202204
[FETCH] 202205
[FETCH] 202206
[FETCH] 202207
[FETCH] 202208
[FETCH] 202209
[FETCH] 202210
[FETCH] 202211
[FETCH] 202212
[FETCH] 202301
[FETCH] 202302
[FETCH] 202303
[FETCH] 202304
[FETCH] 202305
[FETCH] 202306
[FETCH] 202307
[FETCH] 202308
[FETCH] 202309
[FETCH] 202310
[FETCH] 202311
[FETCH] 202312
[FETCH] 202401
[FETCH] 202402
[FETCH] 202403
[FETCH] 202404
[FETCH] 202405
[FETCH] 202406
[FETCH] 202407
[FETCH] 202408
[FETCH] 202409
[FETCH] 202410
[FETCH] 202411
[FETCH] 202412
[FETCH] 202501
[FETCH] 202502
[FETCH] 202503
[FETCH] 202504
[FETCH] 202505
[FETCH] 202506
[FETCH] 20

In [38]:
df_final = pd.concat(
    [df_air.explode("data").drop(columns="data").reset_index(drop=True),
     pd.json_normalize(df_air.explode("data")["data"])],
    axis=1
)

In [40]:
# df_final.to_csv(r'C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KS_Air_Tourism\kac_sample.csv')